# Atelier 4.A — Agent RAG multi-sources ★

**Durée : 90 min** · Projet fil rouge, incrément du Jour 4

L'assistant du Jour 2 connaît la documentation mais ne fait qu'une recherche ; l'agent du Jour 3 consulte le CRM mais ignore la documentation. L'**agent RAG** réunit les deux : la recherche devient un **outil** parmi d'autres, et l'agent décide quelles sources interroger, dans quel ordre, et combien de fois.

Il répond au client connecté au portail de Fictiva à partir de trois sources :

| Source | Contenu | Outils | Référence citée |
|---|---|---|---|
| Documentation (Jour 2) | CGV, retours, livraison, maintenance, catalogue, FAQ, portail | `rechercher_documentation` | `[DOC-12]` |
| Base de connaissances du support | pannes, codes d'erreur, procédures de dépannage | `rechercher_support` | `[KB-104]` |
| CRM (Jour 3) | fiche du client, commandes, calcul de remboursement | `fiche_client`, `mes_commandes`, `consulter_commande`, `calculer_remboursement` | `[CMD-2026-0412]` |

Livrable : l'agent, exposé par le service web du Jour 2 (`POST /agent`). Les briques sont dans `agent_rag.py`.

In [ ]:
import json
import subprocess
import sys
import time

import httpx
import pandas as pd

import agent_rag as ar
import outils_llm as llm
import rag

MODELE = llm.choisir_modele()   # le modèle doit savoir appeler des outils
print("Modèle :", MODELE)

## 1. Les sources de connaissance

La documentation est découpée et indexée comme au Jour 2 (sections, en-têtes, recherche hybride et reranker). La base du support est un **second index** : un article par extrait, avec ses produits. Deux index séparés, c'est le **RAG multi-index** : chaque source garde son vocabulaire, et l'agent choisit où chercher.

In [ ]:
sources = ar.construire_sources()   # premier lancement : téléchargement des modèles (0,8 Go)
print(len(sources.documentation.extraits), "extraits de documentation,", len(sources.support.extraits), "articles de support")
pd.DataFrame(ar.charger_articles())[["id", "titre", "produits", "confidentialite"]]

L'article `KB-900` est une **note interne** : comme la procédure interne du Jour 2, il est filtré **avant** la recherche (`public_seulement=True`). L'agent ne peut pas citer ce qu'il ne reçoit pas.

In [ ]:
for source, requete in [("documentation", "délai de retour d'un article sans défaut"),
                        ("support", "LX-500 code E-21 voyant rouge"),
                        ("support", "défaut connu sur un lot de LX-500")]:
    print(f"== {source} : {requete}")
    for e in sources.rechercher(source, requete):
        print(f"   [{ar.reference(e)}] {e.reference()[:80]}  (score {e.score:.2f})")

## 2. Le RAG naïf face à une question multi-sources

Une cliente de Rochat Emballages écrit : *« Une des imprimantes LX-500 que nous venons de recevoir affiche le code E-21. Est-elle sous garantie et que devons-nous faire ? »*

Il faut trois informations : la date de livraison (CRM), la durée de la garantie (CGV) et la signification du code E-21 (support). L'assistant du Jour 2 fait **une** recherche, dans **une** source :

In [ ]:
QUESTION_E21 = ("Une des imprimantes LX-500 que nous venons de recevoir affiche le code E-21. "
                "Est-elle sous garantie et que devons-nous faire ?")
assistant_j2 = rag.Assistant(sources.documentation, sources.reranker, modele=MODELE)
r = assistant_j2.repondre(QUESTION_E21)
print(r.texte, "\n\nSources :", r.sources())

Au mieux une réponse partielle : la documentation ne connaît ni la commande du client, ni le code E-21.

## 3. Les outils de l'agent

Chaque outil renvoie des extraits ou des fiches **précédés de leur référence**, que l'agent doit citer. Les outils du CRM ne prennent **pas** le numéro de client en paramètre : il vient de la session du portail, transmis dans la configuration de l'exécution (`RunnableConfig`). Le modèle ne le voit pas et ne peut pas le changer : c'est le **moindre privilège** appliqué à l'identité.

In [ ]:
outils = ar.creer_outils(sources)
for o in outils:
    print(f"{o.name}({', '.join(o.args)})")

In [ ]:
consulter = next(o for o in outils if o.name == "consulter_commande")
print(consulter.invoke({"numero_commande": "CMD-2026-0412"}, config=ar.config("CL-10234"))[:300], "\n")
# la même commande, demandée depuis le compte d'un autre client :
print(consulter.invoke({"numero_commande": "CMD-2026-0412"}, config=ar.config("CL-10456")))

Le message d'erreur ne dit pas si la commande existe ailleurs : il ne révèle rien à un curieux.

## 4. L'agent

Même graphe qu'au Jour 3 (`agent ↔ outils`, avec mémoire), mais d'autres outils et un autre prompt. Le prompt donne la **méthode** : repérer les sources nécessaires, ne rien affirmer sans outil, citer les références, refuser hors du domaine.

In [ ]:
print(ar.PROMPT_AGENT)
agent = ar.construire_agent(outils, modele=MODELE)
print(agent.get_graph().draw_mermaid())

Cinq questions de nature différente : documentation seule, support seul, CRM seul, plusieurs sources, hors du domaine. Observez les **trajectoires** : quels outils, dans quel ordre, combien d'appels au modèle.

In [ ]:
QUESTIONS = [
    ("CL-10456", "Livrez-vous au Liechtenstein, et en combien de temps ?"),
    ("CL-10456", "Quand je tape un Z sur mon clavier KB-80, c'est un Y qui s'affiche. Que faire ?"),
    ("CL-10234", "Quand ma dernière commande a-t-elle été livrée ?"),
    ("CL-10234", QUESTION_E21),
    ("CL-10234", "Quel temps fera-t-il demain à Lausanne ?"),
]
resultats_agent = {}
for client, question in QUESTIONS:
    print("=" * 100, f"\n{client} : {question}\n")
    resultats_agent[question] = r = await ar.demander(agent, client, question, f"atelier-{len(resultats_agent)}")
    ar.afficher(r, sources)

In [ ]:
pd.DataFrame([{"question": q[:60], "sources": ", ".join(r.sources_utilisees()) or "aucune",
               "appels_modele": r.appels_modele, "appels_outils": len(r.outils), "tokens": r.tokens,
               "latence_s": r.latence_s} for q, r in resultats_agent.items()])

**À observer.** Pour la météo, l'agent n'appelle aucun outil. Pour la question sur le code E-21, il devrait interroger les trois sources : vérifiez dans sa trajectoire qu'il a bien consulté la date de livraison. S'il s'en est passé, sa réponse sur la garantie reste générale (« garanti 24 mois ») au lieu de dire si **cette** imprimante est couverte : c'est une erreur de trajectoire, que l'atelier 4.C mesure (rappel des sources). Le nombre d'appels au modèle varie avec la difficulté : c'est la force de l'agent (il s'adapte) et son coût (latence et tokens).

## 5. Des citations vérifiables

Une citation n'a de valeur que si elle renvoie à un extrait réellement transmis à l'agent. On compare les références **citées** dans la réponse et les références **renvoyées** par les outils : une citation qui ne correspond à aucun résultat d'outil est inventée.

In [ ]:
r = resultats_agent[QUESTION_E21]
print("Renvoyées par les outils :", r.renvoyees)
print("Citées dans la réponse   :", r.citees)
print("Citations inventées      :", r.invalides or "aucune")

In [ ]:
# Une réponse fabriquée pour l'exemple : [DOC-58] n'a été renvoyé par aucun outil.
fausse = "Votre imprimante est garantie 36 mois [DOC-58] et le code E-21 indique un bourrage [KB-104]."
print("Citations inventées :", [ref for ref in ar.references_citees(fausse) if ref not in r.renvoyees])

Le contrôle repère une **référence inventée**, pas une **affirmation fausse** attribuée à une vraie référence (« bourrage » avec [KB-104]) : pour cela, il faut un juge de fidélité, que l'on mettra en place dans l'atelier 4.C.

## 6. Une conversation sur plusieurs tours

La mémoire (*checkpointer*) conserve les messages de chaque conversation (`thread_id`). L'agent peut répondre à « ces claviers » sans que la cliente répète le numéro de commande. Le fil de conversation est préfixé par le numéro de client : un client ne peut pas reprendre la conversation d'un autre.

In [ ]:
TOURS = ["Quelle est ma dernière commande ?",
         "Si je retourne ces claviers sans défaut, combien serai-je remboursée ?",
         "Et qui paie l'envoi du retour ?"]
for message in TOURS:
    print("=" * 100, f"\n👤 {message}\n")
    ar.afficher(await ar.demander(agent, "CL-10456", message, "conversation-claviers"), sources)

In [ ]:
etat = await agent.aget_state(ar.config("CL-10456", "conversation-claviers"))
print(len(etat.values["messages"]), "messages en mémoire dans cette conversation")
# Une nouvelle conversation repart de zéro :
ar.afficher(await ar.demander(agent, "CL-10456", "Et qui paie l'envoi du retour ?", "nouvelle-conversation"))

Le contexte grandit à chaque tour : sur une longue conversation, on résume les anciens tours ou on ne garde que les derniers messages.

## 7. Routeur explicite ou agent ? (Adaptive RAG)

Un agent coûte plusieurs appels au modèle, même pour une question simple. L'**Adaptive RAG** place un **routeur** en amont : un appel de classification choisit la stratégie la moins coûteuse.

| Stratégie | Traitement | Appels au modèle |
|---|---|---|
| `documentation` ou `support` | une recherche, une génération citée (RAG simple) | 2 (routeur + génération) |
| `agent` | l'agent multi-sources | 1 + ceux de l'agent |
| `hors_perimetre` | réponse fixe, sans recherche | 1 |

In [ ]:
async def repondre_adaptatif(client: str, question: str) -> dict:
    debut = time.perf_counter()
    routage = ar.router(question, MODELE)
    appels = 1
    if routage.strategie in ("documentation", "support"):
        extraits = sources.rechercher(routage.strategie, question)
        texte, appels = ar.generer(question, extraits, MODELE).texte, appels + 1
    elif routage.strategie == "hors_perimetre":
        texte = ar.REPONSE_HORS_PERIMETRE
    else:
        r = await ar.demander(agent, client, question, f"adaptatif-{time.time()}")
        texte, appels = r.reponse, appels + r.appels_modele
    return {"strategie": routage.strategie, "justification": routage.justification, "reponse": texte,
            "appels_modele": appels, "latence_s": round(time.perf_counter() - debut, 1)}

comparaison = []
for client, question in QUESTIONS:
    adaptatif = await repondre_adaptatif(client, question)
    seul = resultats_agent[question]
    comparaison.append({"question": question[:55], "strategie": adaptatif["strategie"],
                        "appels (routeur)": adaptatif["appels_modele"], "appels (agent seul)": seul.appels_modele,
                        "latence (routeur)": adaptatif["latence_s"], "latence (agent seul)": seul.latence_s})
    print(f"{adaptatif['strategie']:>15} | {question[:55]}\n                  {adaptatif['reponse'][:160]}\n")
pd.DataFrame(comparaison)

**À discuter.** Le routeur économise des appels sur les questions simples, mais une **erreur de routage est définitive** : une question classée « documentation » qui demandait le CRM obtiendra une réponse incomplète. L'agent seul est plus robuste, mais plus cher. On choisit en mesurant les deux sur le golden set (atelier 4.C).

## 8. Exposer l'agent dans l'API du Jour 2

Le service web du Jour 2 (`api_assistant.py`) garde sa route `POST /question` et gagne `POST /agent`. La requête porte le numéro du client connecté (en production, il viendrait du jeton d'authentification du portail) et, pour poursuivre une conversation, l'identifiant renvoyé par la réponse précédente. Le service applique en plus les garde-fous de l'atelier 4.C.

In [ ]:
serveur = subprocess.Popen([sys.executable, "-m", "uvicorn", "api_assistant:app", "--port", "8002"],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(120):   # le service construit ses index au démarrage
    try:
        print(httpx.get("http://localhost:8002/sante", timeout=2).json())
        break
    except httpx.HTTPError:
        time.sleep(1)

In [ ]:
reponse = httpx.post("http://localhost:8002/agent", timeout=600, json={
    "numero_client": "CL-10901",
    "message": "Notre NAS NS-4 affiche un voyant orange. Est-il encore sous garantie ?"}).json()
print(json.dumps(reponse, ensure_ascii=False, indent=2))

In [ ]:
suite = httpx.post("http://localhost:8002/agent", timeout=600, json={
    "numero_client": "CL-10901", "conversation": reponse["conversation"],
    "message": "Combien coûterait un contrat de maintenance Or pour ce serveur ?"}).json()
print(suite["reponse"], "\n\nCitations :", [c["reference"] for c in suite["citations"]], "| outils :", suite["outils"])

In [ ]:
serveur.terminate()

✏️ Ouvrez http://localhost:8002/docs pendant que le service tourne : la route `/agent` y est documentée avec ses schémas d'entrée et de sortie.

## À retenir

- Dans un agent RAG, la **recherche est un outil** : l'agent choisit les sources, reformule, recherche à nouveau si besoin.
- Plusieurs **index séparés** (documentation, support) et des données structurées (CRM) : chaque source garde ses filtres d'accès.
- L'identité du client passe par la **configuration**, jamais par le modèle.
- Des **références citables** dans chaque résultat d'outil rendent les citations vérifiables.
- Un **routeur** (Adaptive RAG) réduit le coût des questions simples, au prix d'erreurs de routage définitives.

## Pour aller plus loin

- Ajoutez un outil `rechercher_tickets` sur un historique de tickets résolus (une quatrième source).
- Remplacez les outils CRM locaux par le serveur MCP du Jour 3, en passant l'identité du client par l'authentification du serveur.